# Convert MIDI piece to PIG format

Some relevant constants:

In [1]:
import os
import pandas as pd

# RELEVANT CONSTANTS
PATH_TO_DATASET_FOLDER = './PianoFingeringDataset_v1.2/PianoFingeringDataset_v1.2/FingeringFiles'
PATH_TO_METADATA = './PianoFingeringDataset_v1.2/PianoFingeringDataset_v1.2/List.csv'
directory_path = PATH_TO_DATASET_FOLDER

spelled_pitch_values = set()

for filename in os.listdir(directory_path):
    file_path = os.path.join(directory_path, filename)
    if os.path.isfile(file_path):
        df = pd.read_table(file_path, sep="\t", skiprows=1, names=["noteID", "onset_time", "offset_time", "spelled_pitch", "onset_velocity", "offset_velocity", "channel", "finger_number"])
        spelled_pitch_values.update(df['spelled_pitch'].unique())


spelled_pitch_values = sorted(spelled_pitch_values)    # convert "spelled pitch" field to a number: create the mapping in the first place
pitch_to_int_mapping = {p:i for i, p in enumerate(spelled_pitch_values)}

print(f"pitch_to_int_mapping: {pitch_to_int_mapping}")

print("\n")

# We need to do the same with the fingerings themselves, since the dataset includes fingerings like "-1_-2" to denote finger changes on the same note
fingering_map = set()

for filename in os.listdir(directory_path):
    file_path = os.path.join(directory_path, filename)
    if os.path.isfile(file_path):
        df = pd.read_table(file_path, sep="\t", skiprows=1, names=["noteID", "onset_time", "offset_time", "spelled_pitch", "onset_velocity", "offset_velocity", "channel", "finger_number"])
        fingering_map.update(df['finger_number'].unique())

finger_to_int_mapping = {p:i for i, p in enumerate(fingering_map)}    # convert "spelled pitch" field to a number: create the mapping in the first place
print(f"finger_to_int_mapping: {finger_to_int_mapping}")


pitch_to_int_mapping:  {'Ab0': 0, 'A0': 1, 'A#0': 2, 'Bb0': 3, 'B0': 4, 'B#0': 5, 'Cb0': 6, 'C0': 7, 'C#0': 8, 'Db0': 9, 'D0': 10, 'D#0': 11, 'Eb0': 12, 'E0': 13, 'E#0': 14, 'Fb0': 15, 'F0': 16, 'F#0': 17, 'Gb0': 18, 'G0': 19, 'G#0': 20, 'Ab1': 21, 'A1': 22, 'A#1': 23, 'Bb1': 24, 'B1': 25, 'B#1': 26, 'Cb1': 27, 'C1': 28, 'C#1': 29, 'Db1': 30, 'D1': 31, 'D#1': 32, 'Eb1': 33, 'E1': 34, 'E#1': 35, 'Fb1': 36, 'F1': 37, 'F#1': 38, 'Gb1': 39, 'G1': 40, 'G#1': 41, 'Ab2': 42, 'A2': 43, 'A#2': 44, 'Bb2': 45, 'B2': 46, 'B#2': 47, 'Cb2': 48, 'C2': 49, 'C#2': 50, 'Db2': 51, 'D2': 52, 'D#2': 53, 'Eb2': 54, 'E2': 55, 'E#2': 56, 'Fb2': 57, 'F2': 58, 'F#2': 59, 'Gb2': 60, 'G2': 61, 'G#2': 62, 'Ab3': 63, 'A3': 64, 'A#3': 65, 'Bb3': 66, 'B3': 67, 'B#3': 68, 'Cb3': 69, 'C3': 70, 'C#3': 71, 'Db3': 72, 'D3': 73, 'D#3': 74, 'Eb3': 75, 'E3': 76, 'E#3': 77, 'Fb3': 78, 'F3': 79, 'F#3': 80, 'Gb3': 81, 'G3': 82, 'G#3': 83, 'Ab4': 84, 'A4': 85, 'A#4': 86, 'Bb4': 87, 'B4': 88, 'B#4': 89, 'Cb4': 90, 'C4': 91, 'C#4'

In [38]:
from music21 import converter, midi, note, chord
import pandas as pd

def midi_to_pig_format(midi_file, output_txt="output_pig_format.csv"):
    # Parse MIDI file
    score = converter.parse(midi_file)

    # Extract MIDI parts (assume first track is RH, second is LH if multi-track)
    parts = score.parts
    is_multitrack = len(parts) > 1

    if not is_multitrack:
        print("NOT MULTITRACK!")
        return

    note_data = []
    note_id = 0  # Sequential numbering

    for part_index, part in enumerate(parts):        
        # Assign hand based on track index (if multitrack, first track is RH (0), second is LH (1))
        hand = 0 if not is_multitrack or part_index == 0 else 1

        for element in part.flat.notes:
            if isinstance(element, note.Note):  # Single Note
                spelled_pitch = element.nameWithOctave
                onset_time = element.offset  # Start time in quarter notes
                offset_time = onset_time + element.quarterLength  # End time

                # MIDI velocity (MIDI files may not always contain explicit velocity info)
                onset_velocity = element.volume.velocity or 64  # Default to 64 if missing
                offset_velocity = onset_velocity  # No explicit release velocity in MIDI

                note_data.append([
                    note_id, onset_time, offset_time, spelled_pitch,
                    onset_velocity, offset_velocity, hand
                ])
                note_id += 1

            elif isinstance(element, chord.Chord):  # Chord (multiple notes at same time)
                for pitch in element.pitches:
                    spelled_pitch = pitch.nameWithOctave
                    onset_time = element.offset
                    offset_time = onset_time + element.quarterLength
                    onset_velocity = element.volume.velocity or 64
                    offset_velocity = onset_velocity

                    note_data.append([
                        note_id, onset_time, offset_time, spelled_pitch,
                        onset_velocity, offset_velocity, hand
                    ])
                    note_id += 1

    # Convert to DataFrame
    df = pd.DataFrame(note_data, columns=[
        "noteID", "onset_time", "offset_time", "spelled_pitch",
        "onset_velocity", "offset_velocity", "channel"
    ])

    # Sort by onset time
    df = df.sort_values(by="onset_time").reset_index(drop=True)


    # IMPORTANT: ONLY DOES A SPECIFIC HAND
    df = df[df["channel"] == 0].reset_index(drop=True)   # 0 is right, 1 is left


    # Save to CSV
    with open(output_txt, "w") as f:
        # Write header
        f.write("noteID onset_time offset_time spelled_pitch onset_velocity offset_velocity channel\n")
        # Write each row
        for _, row in df.iterrows():
            f.write(f"{row['noteID']} {row['onset_time']} {row['offset_time']} {row['spelled_pitch']} "
                    f"{row['onset_velocity']} {row['offset_velocity']} {row['channel']}\n")

    print(f"PIG-formatted text file saved as {output_txt}")

    return df



# CONVERTS MIDI FILE TO CSV
midi_file = "records.mid"  # Replace with your MIDI file path
df_pig = midi_to_pig_format(midi_file, "output_pig_format.txt")


PIG-formatted text file saved as output_pig_format.txt


/home/willy/Desktop/projects/smolgrad/smol/lib/python3.12/site-packages/music21/stream/base.py:3689: Music21DeprecationWarning: .flat is deprecated.  Call .flatten() instead
  return self.iter().getElementsByClass(classFilterList)


In [39]:
pd.read_csv("output_pig_format.txt")

,noteID onset_time offset_time spelled_pitch onset_velocity offset_velocity channel
0,0 0.5 1.0 C#5 80 80 0
1,1 1.0 1.5 C#5 80 80 0
2,2 1.5 2.0 B4 80 80 0
3,3 2.0 4.0 C#5 80 80 0
4,4 4.0 4.5 C#5 80 80 0
...,...
390,390 284.5 286.5 A4 80 80 0
391,391 286.5 287.0 B4 80 80 0
392,392 287.0 287.5 F#4 80 80 0
393,393 287.5 288.0 A4 80 80 0


# Load the Relevant Model

In [23]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

directory_path = './PianoFingeringDataset_v1.2/PianoFingeringDataset_v1.2/FingeringFiles/'

all_sequences = []   # will hold raw DataFrames for each piece
all_piece_names = []

pitch_vocab = set()
finger_vocab = set()

FINGERING_TYPE_TO_ANALYZE = 1

# 1) Gather data by piece
for filename in os.listdir(directory_path):
    file_path = os.path.join(directory_path, filename)

    # Example: "001-01_fingering.txt" => fingering_label = "001-01", rest = "fingering.txt"
    fingering_label, _ = filename.split('_')  # e.g. "001-01"
    piece_id, fingering_type = fingering_label.split('-')

    # Check if this is a valid file for the desired fingering type
    if os.path.isfile(file_path) and FINGERING_TYPE_TO_ANATLYZE == int(fingering_type):
    # if os.path.isfile(file_path):
        df = pd.read_table(
            file_path, sep="\t", skiprows=1,
            names=[
                "noteID", "onset_time", "offset_time", "spelled_pitch",
                "onset_velocity", "offset_velocity", "channel",
                "finger_number"
            ]
        )

        # Update global vocab sets
        pitch_vocab.update(df["spelled_pitch"].unique())
        finger_vocab.update(map(str, df["finger_number"].unique()))

        all_piece_names.append(filename)
        all_sequences.append(df)

# 2) Build the pitch & finger mappings
pitch_vocab = sorted(pitch_vocab)
pitch_to_idx = {p: i for i, p in enumerate(pitch_vocab)}

finger_vocab = sorted(finger_vocab)
finger_to_idx = {f: i for i, f in enumerate(finger_vocab)}

print(f"Total number of pieces found: {len(all_sequences)}")

# 3) Convert each piece’s DF into (X_list, y_list) with shape (num_notes, 6) + (num_notes,)
raw_encoded_sequences = []  # We'll store the unscaled version first

for df in all_sequences:
    X_list = []
    y_list = []

    for row in df.itertuples(index=False):
        spelled_pitch = row.spelled_pitch
        finger_str = str(row.finger_number)

        # Encode spelled pitch
        pitch_int = pitch_to_idx.get(spelled_pitch, 0)  # fallback 0 if unseen

        # Encode finger
        finger_int = finger_to_idx.get(finger_str, 0)   # fallback 0 if unseen

        # Construct feature row:
        # [pitch_int, onset_time, offset_time, onset_velocity, offset_velocity, channel]
        feature_row = [
            pitch_int,
            float(row.onset_time),
            float(row.offset_time),
            float(row.onset_velocity),
            float(row.offset_velocity),
            float(row.channel)
        ]
        X_list.append(feature_row)
        y_list.append(finger_int)

    raw_encoded_sequences.append((X_list, y_list))

# 3.5) Gather all numeric features for scaling
# We'll do a single pass over raw_encoded_sequences
numeric_data = []
for (X_list, _) in raw_encoded_sequences:
    for row in X_list:
        numeric_vals = row[1:]  # 5 numeric features => (onset_time, offset_time, onset_vel, offset_vel, channel)
        numeric_data.append(numeric_vals)

numeric_data = np.array(numeric_data, dtype=np.float32)
print("numeric_data shape:", numeric_data.shape)

# 3.6) Fit a StandardScaler on the numeric columns
scaler = StandardScaler()
scaler.fit(numeric_data)
print("Numeric scaler mean_:", scaler.mean_)
print("Numeric scaler var_ :", scaler.var_)

# 3.7) Apply the scaler to each piece
scaled_sequences = []
for (X_list, y_list) in raw_encoded_sequences:
    new_X_list = []
    for row in X_list:
        pitch_val = row[0]
        numeric_vals = row[1:]  # shape (5,)
        numeric_vals_arr = np.array(numeric_vals, dtype=np.float32).reshape(1, -1)
        scaled_vals = scaler.transform(numeric_vals_arr)[0]  # (5,)

        # Rebuild the row: pitch_val + scaled numeric
        new_row = [pitch_val] + scaled_vals.tolist()
        new_X_list.append(new_row)
    scaled_sequences.append((new_X_list, y_list))

# 4) Split entire set of sequences (by piece) into train/val/test
train_val_seqs, test_seqs = train_test_split(
    scaled_sequences, test_size=0.2, random_state=42
)
train_seqs, val_seqs = train_test_split(
    train_val_seqs, test_size=0.25, random_state=42
)

print(f"Total pieces: {len(scaled_sequences)}")
print(f"Train pieces: {len(train_seqs)}")
print(f"Val pieces:   {len(val_seqs)}")
print(f"Test pieces:  {len(test_seqs)}")

# Each item is now (X, y) with scaled numeric features


Total number of pieces found: 148
numeric_data shape: (45976, 5)
Numeric scaler mean_: [21.13948002 21.45820221 68.73305638 79.79739429  0.45321472]
Numeric scaler var_ : [3.97946937e+02 3.98278724e+02 3.89274639e+02 1.63700132e+01
 2.47811138e-01]
Total pieces: 148
Train pieces: 88
Val pieces:   30
Test pieces:  30


In [8]:
# RNN MODEL
class PianoRNNModel(nn.Module):
    def __init__(
        self,
        num_pitches,     # size of pitch vocab
        num_fingers,     # number of finger classes
        embed_dim=32,    # dimension for pitch embedding
        hidden_dim=64,   # dimension for RNN hidden layer
        numeric_dim=5,   # number of extra numeric features
    ):
        super().__init__()

        # have 2 embedding vectors: one for pitch and the other numeric features. We then concatenate them and that ends up being the hidden state of our RNN.
        self.embedding_pitch = nn.Embedding(num_pitches, embed_dim)        # Embedding for pitch (categorical)
        self.embedding_numeric = nn.Linear(numeric_dim, embed_dim)     # A simple linear projection for numeric features => same embed_dim so we can concatenate them with pitch embeddings

        self.rnn = nn.RNN(embed_dim * 2, hidden_dim, batch_first=True)         # RNN: input_dim = embed_dim(pitch) + embed_dim(numeric) = 2*embed_dim
        self.fc = nn.Linear(hidden_dim, num_fingers)   # Final linear layer to map hidden_dim => number of finger classes

    def forward(self, x):
        pitch_ids = x[..., 0].long()         # shape => (batch_size, seq_len)
        numeric_feats = x[..., 1:].float()   # shape => (batch_size, seq_len, 5)

        # Embed pitch and numeric separately
        pitch_emb = self.embedding_pitch(pitch_ids)  # (batch_size, seq_len, embed_dim)
        numeric_emb = self.embedding_numeric(numeric_feats)  # (batch_size, seq_len, embed_dim)

        # Concatenate along last dimension => (batch_size, seq_len, 2*embed_dim)
        combined_emb = torch.cat((pitch_emb, numeric_emb), dim=2)

        # RNN => (batch_size, seq_len, hidden_dim)
        rnn_out, _ = self.rnn(combined_emb)

        # Final linear => (batch_size, seq_len, num_fingers)
        logits = self.fc(rnn_out)
        return logits

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score

def train_sequence_model(model, train_loader, val_loader, num_epochs=5, lr=1e-3):
    """
    Trains a token-level sequence model on multi-feature input X_seq.
    Expects X_seq.shape = (batch_size, seq_len, num_features)
           y_seq.shape = (batch_size, seq_len)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()  # for token-level classification

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for X_seq, y_seq in train_loader:
            # X_seq shape: (batch_size, seq_len, num_features)
            # y_seq shape: (batch_size, seq_len)

            X_seq = X_seq.to(device, non_blocking=True)
            y_seq = y_seq.to(device, non_blocking=True)

            optimizer.zero_grad()

            # Forward pass: model should output shape (batch_size, seq_len, num_fingers)
            logits = model(X_seq)

            # Reshape for CrossEntropy:
            # Flatten tokens => (batch_size * seq_len, num_fingers)
            logits_reshaped = logits.view(-1, logits.size(-1))
            y_seq_reshaped = y_seq.view(-1)

            loss = criterion(logits_reshaped, y_seq_reshaped)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Evaluate on validation set
        val_acc = evaluate_sequence_model(model, val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | ValAcc: {val_acc:.4f}")

def evaluate_sequence_model(model, data_loader):
    """
    Evaluates token-level accuracy. Expects same shapes as train_sequence_model.
    """
    device = next(model.parameters()).device
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_seq, y_seq in data_loader:
            # X_seq: (batch_size, seq_len, num_features)
            # y_seq: (batch_size, seq_len)

            X_seq = X_seq.to(device, non_blocking=True)
            y_seq = y_seq.to(device, non_blocking=True)

            logits = model(X_seq)  # => (batch_size, seq_len, num_fingers)
            preds = torch.argmax(logits, dim=-1)  # => (batch_size, seq_len)

            all_preds.extend(preds.view(-1).cpu().numpy())
            all_labels.extend(y_seq.view(-1).cpu().numpy())

    return accuracy_score(all_labels, all_preds)


In [64]:

import torch
from torch.utils.data import Dataset, DataLoader

class FingeringDataset(Dataset):
    def __init__(self, sequences):
        """
        sequences: list of (X_list, y_list),
                   where X_list is shape (seq_len, 6)
                         y_list is shape (seq_len,)
        """
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        X_list, y_list = self.sequences[idx]
        # Convert to Tensors
        # By now, X_list is already scaled
        X_tensor = torch.tensor(X_list, dtype=torch.float32)  # (seq_len, 6)
        y_tensor = torch.tensor(y_list, dtype=torch.long)     # (seq_len,)
        return X_tensor, y_tensor

# Create dataset objects
train_dataset = FingeringDataset(train_seqs)
val_dataset   = FingeringDataset(val_seqs)
test_dataset  = FingeringDataset(test_seqs)

# If your model can handle variable sequence lengths with e.g. a collate_fn,
# you can do batch_size > 1. Otherwise, keep batch_size=1.
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)


# Create your model instance
rnn_model = MultiFeatureRNNTagger(
    num_pitches=len(pitch_vocab),
    num_fingers=len(finger_vocab),
    embed_dim=32,
    hidden_dim=64,
    numeric_dim=5
)

# Train as usual
train_sequence_model(rnn_model, train_loader, val_loader, num_epochs=10, lr=1e-3)
test_acc_rnn = evaluate_sequence_model(rnn_model, test_loader)
print(f"RNN Test Accuracy = {test_acc_rnn:.4f}")

Epoch 1/10 | Loss: 2.4655 | ValAcc: 0.3038
Epoch 2/10 | Loss: 1.6684 | ValAcc: 0.3596
Epoch 3/10 | Loss: 1.5610 | ValAcc: 0.3710
Epoch 4/10 | Loss: 1.5155 | ValAcc: 0.3867
Epoch 5/10 | Loss: 1.4763 | ValAcc: 0.3976
Epoch 6/10 | Loss: 1.4515 | ValAcc: 0.3996
Epoch 7/10 | Loss: 1.4261 | ValAcc: 0.3900
Epoch 8/10 | Loss: 1.4014 | ValAcc: 0.4269
Epoch 9/10 | Loss: 1.3721 | ValAcc: 0.4345
Epoch 10/10 | Loss: 1.3413 | ValAcc: 0.4445
RNN Test Accuracy = 0.4613


In [24]:
import os
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score


def predict_piece_fingerings_multi(
    model,
    file_path,
    pitch_to_idx,         # dict mapping spelled pitches -> int
    scaler=None,          # the numeric scaler fit during training
    finger_to_idx=None,   # dict mapping finger labels -> int (optional)
    idx_to_finger=None,   # reverse mapping to decode predictions (optional)
    device=None
):
    """
    Predict fingerings for a single piece using a trained multi-feature RNN/LSTM/Transformer model.
    Applies the same numeric scaling used during training if `scaler` is provided.

    Returns:
      - preds:        (seq_len,) integer array of predicted finger indices
      - y_true:       (seq_len,) integer array of actual finger indices (or None if finger_to_idx not given)
      - y_pred_labels: decoded predicted finger labels (list of str) or the raw preds
      - y_true_labels: decoded actual finger labels (list of str) or None
      - acc:          float, accuracy on this piece if y_true is available, else None
    """

    # 1. Load piece data
    df = pd.read_table(
        file_path,
        sep=" ",
        skiprows=1, #noteID onset_time offset_time spelled_pitch onset_velocity offset_velocity channel
        names=[
            "noteID",
            "onset_time",
            "offset_time",
            "spelled_pitch",
            "onset_velocity",
            "offset_velocity",
            "channel",
        ],
    )

    # 2. Encode spelled pitch -> int
    df["spelled_pitch_int"] = df["spelled_pitch"].map(pitch_to_idx).fillna(0).astype(int)

  
    # 4. Construct multi-feature array X of shape (seq_len, 6)
    #    X[..., 0] = spelled_pitch_int (categorical)
    #    X[..., 1] = onset_time
    #    X[..., 2] = offset_time
    #    X[..., 3] = onset_velocity
    #    X[..., 4] = offset_velocity
    #    X[..., 5] = channel
    seq_len = len(df)
    X_array = np.zeros((seq_len, 6), dtype=np.float32)
    print(df)

    # Fill in pitch as int (col 0).
    # We'll keep it as float32 for consistency, then convert to long in the model if needed.
    X_array[:, 0] = df["spelled_pitch_int"].astype(float)

    # Fill in numeric features (cols 1..5)
    X_array[:, 1] = df["onset_time"].astype(float)
    X_array[:, 2] = df["offset_time"].astype(float)
    X_array[:, 3] = df["onset_velocity"].astype(float)
    X_array[:, 4] = df["offset_velocity"].astype(float)
    X_array[:, 5] = df["channel"].astype(float)

    # 5. Apply the same scaler as training (if provided) to columns 1..5
    if scaler is not None:
        pitch_column = X_array[:, 0].copy()  # store pitch in a temp var
        numeric_vals = X_array[:, 1:]       # shape => (seq_len, 5)

        # Scale numeric features => shape (seq_len, 5)
        numeric_vals_scaled = scaler.transform(numeric_vals)

        # Rebuild X_array with pitch + scaled numeric
        X_array = np.concatenate([
            pitch_column.reshape(-1, 1),  # shape => (seq_len, 1)
            numeric_vals_scaled
        ], axis=1)  # => shape => (seq_len, 6)

    # 6. Convert to Torch tensor: shape => (1, seq_len, 6)
    X_tensor = torch.tensor(X_array, dtype=torch.float32).unsqueeze(0)

    # 7. Move model & input to device
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_tensor = X_tensor.to(device)
    model = model.to(device)
    model.eval()

    # 8. Inference: model output => (1, seq_len, num_fingers)
    with torch.no_grad():
        logits = model(X_tensor)
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy()  # => (seq_len,)

    # 9. Decode predictions (optional)
    y_pred_labels = preds

    # 10. Compute accuracy if y_true is available
    acc = 0

    return preds, 0, y_pred_labels, 0, 0


piece_file = "output_pig_format.txt"
file_path = piece_file

# Suppose your pitch_to_idx, finger_to_idx, idx_to_finger are defined
# and your model is a MultiFeatureTransformerTagger or MultiFeatureRNNTagger, etc.

# ***IMPORTANT***: Use the SAME scaler you used in training.
# e.g. from the step: `scaler = StandardScaler(); scaler.fit(...)`
# or you saved it in joblib/pickle.

rnn_model = PianoRNNModel(
    num_pitches=len(pitch_to_int_mapping),
    num_fingers=len(finger_to_int_mapping),
    embed_dim=32,
    hidden_dim=64,
    numeric_dim=5
)

state_dict = torch.load('rnn_model_weights.pth')
rnn_model.load_state_dict(state_dict)

rnn_preds, rnn_true, rnn_pred_labels, rnn_true_labels, rnn_acc = predict_piece_fingerings_multi(
    model=rnn_model,
    file_path=file_path,
    pitch_to_idx=pitch_to_int_mapping,
    scaler=None,  # pass the trained scaler
    finger_to_idx=finger_to_int_mapping,
    idx_to_finger=None
)


print(f"\n=== Multi-Feature RNN Predictions for {piece_file} ===")
print("Predicted fingerings (string labels):", rnn_pred_labels)


     noteID  onset_time  offset_time spelled_pitch  onset_velocity  \
0         0         0.5         1.00           C#5              80   
1         1         1.0         1.50           C#5              80   
2         2         1.5         2.00            B4              80   
3         3         2.0         4.00           C#5              80   
4         4         4.0         4.50           C#5              80   
..      ...         ...          ...           ...             ...   
390     390       284.5       286.50            A4              80   
391     391       286.5       287.00            B4              80   
392     392       287.0       287.50           F#4              80   
393     393       287.5       288.00            A4              80   
394     394       288.0       291.75            A4              80   

     offset_velocity  channel  spelled_pitch_int  
0                 80        0                113  
1                 80        0                113  
2     

/tmp/ipykernel_5248/942518143.py:127: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('rnn_model_weights.pth')


In [32]:
# LSTM PREDS INSTEAD:
class LSTMModel(nn.Module):
    def __init__(
        self,
        num_pitches,     # size of pitch vocab
        num_fingers,     # number of finger classes
        embed_dim=32,    # dimension for pitch embedding
        hidden_dim=64,   # dimension for LSTM hidden layer
        numeric_dim=5,   # number of extra numeric features
    ):
        super().__init__()
        self.embedding_pitch = nn.Embedding(num_pitches, embed_dim)    # Embedding for pitch (categorical)
        self.embedding_numeric = nn.Linear(numeric_dim, embed_dim)     # Linear projection for numeric features
        self.lstm = nn.LSTM(embed_dim * 2, hidden_dim, batch_first=True)    # LSTM: input_dim = embed_dim(pitch) + embed_dim(numeric) = 2*embed_dim
        self.fc = nn.Linear(hidden_dim, num_fingers)     # Final linear layer to the num_fingers

    def forward(self, x):
        pitch_ids = x[..., 0].long()
        numeric_feats = x[..., 1:].float()

        # Embed pitch and fingering fields
        pitch_emb = self.embedding_pitch(pitch_ids)    # (batch_size, seq_len, embed_dim)
        numeric_emb = self.embedding_numeric(numeric_feats)  # (batch_size, seq_len, embed_dim)

        # Concatenate => (batch_size, seq_len, 2*embed_dim)
        combined_emb = torch.cat((pitch_emb, numeric_emb), dim=2)

        # LSTM => (batch_size, seq_len, hidden_dim)
        lstm_out, _ = self.lstm(combined_emb)

        # Final linear => (batch_size, seq_len, num_fingers)
        logits = self.fc(lstm_out)
        return logits


lstm_model = LSTMModel(
    num_pitches=len(pitch_to_int_mapping),
    num_fingers=len(finger_to_int_mapping),
    embed_dim=32,
    hidden_dim=64,
    numeric_dim=5
)

state_dict = torch.load('lstm_model_weights.pth')
lstm_model.load_state_dict(state_dict)

lstm_preds, lstm_true, lstm_pred_labels, lstm_true_labels, lstm_acc = predict_piece_fingerings_multi(
    model=lstm_model,
    file_path=file_path,
    pitch_to_idx=pitch_to_int_mapping,
    finger_to_idx=finger_to_int_mapping,
    idx_to_finger=int_to_finger_mapping
)

print(f"\n=== Multi-Feature LSTM Predictions for {piece_file} ===")
print("Predicted fingerings (string labels):", lstm_preds)

     noteID  onset_time  offset_time spelled_pitch  onset_velocity  \
0         0         0.5         1.00           C#5              80   
1         1         1.0         1.50           C#5              80   
2         2         1.5         2.00            B4              80   
3         3         2.0         4.00           C#5              80   
4         4         4.0         4.50           C#5              80   
..      ...         ...          ...           ...             ...   
390     390       284.5       286.50            A4              80   
391     391       286.5       287.00            B4              80   
392     392       287.0       287.50           F#4              80   
393     393       287.5       288.00            A4              80   
394     394       288.0       291.75            A4              80   

     offset_velocity  channel  spelled_pitch_int  
0                 80        0                113  
1                 80        0                113  
2     

/tmp/ipykernel_5248/3741846680.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('lstm_model_weights.pth')


In [44]:
# TRANSFORMERS

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        """
        d_model: embedding size
        max_len: maximum sequence length
        """
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        x shape: (seq_len, batch_size, d_model)
        Returns x + positional_encoding
        """
        seq_len = x.size(0)
        # Add the positional encoding to the input
        # shape => (seq_len, batch_size, d_model)
        x = x + self.pe[:seq_len, :].unsqueeze(1)
        return x
class MultiFeatureTransformerTagger(nn.Module):
    def __init__(
        self,
        num_pitches,      # vocabulary size for spelled_pitch
        num_fingers,      # number of finger classes
        pitch_emb_dim=64, # embedding dimension for pitch
        numeric_dim=5,    # number of numeric features (time, velocity, etc.)
        numeric_emb_dim=64,  # dimension to project numeric features
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1
    ):
        super().__init__()

        self.pitch_embedding = nn.Embedding(num_pitches, pitch_emb_dim)      # pitch to embedding
        self.numeric_projection = nn.Linear(numeric_dim, numeric_emb_dim)    # all other numeric features => numeric_emb_dim

        self.d_model = pitch_emb_dim + numeric_emb_dim        # total embedding size = pitch_emb_dim + numeric_emb_dim
        self.pos_encoder = PositionalEncoding(d_model=self.d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=False  # (seq, batch, feature)
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Final linear layer: map each token embedding -> finger label
        self.fc = nn.Linear(self.d_model, num_fingers)

    def forward(self, x):
        """
        x shape: (batch_size, seq_len, 6)
          - x[..., 0]: pitch (int)
          - x[..., 1:]: numeric feats (float)
        Returns: (batch_size, seq_len, num_fingers)
        """
        pitch_ids = x[..., 0].long()        # (batch_size, seq_len)
        numeric_feats = x[..., 1:].float()  # (batch_size, seq_len, 5)

        # pitch_emb => (batch_size, seq_len, pitch_emb_dim)
        pitch_emb = self.pitch_embedding(pitch_ids)

        # numeric_emb => (batch_size, seq_len, numeric_emb_dim)
        numeric_emb = self.numeric_projection(numeric_feats)

        # combine => (batch_size, seq_len, d_model)
        combined_emb = torch.cat((pitch_emb, numeric_emb), dim=2)

        # transform to shape (seq_len, batch_size, d_model) for the Transformer
        combined_emb = combined_emb.transpose(0, 1)  # => (seq_len, batch_size, d_model)

        encoded = self.pos_encoder(combined_emb)

        transformer_out = self.transformer_encoder(encoded)  # => (seq_len, batch_size, d_model)

        logits = self.fc(transformer_out)
        logits = logits.transpose(0, 1)  # => (batch_size, seq_len, num_fingers)
        return logits


In [47]:
###############################################
# Train Transformer
###############################################
#   pitch_vocab = sorted set of pitches
#   finger_vocab = sorted set of finger labels
#   train_loader, val_loader, test_loader
#   Each batch => (X_seq, y_seq) with X_seq.shape=(batch_size, seq_len, 6)
import math
from torch.optim import Adam

transformer_model = MultiFeatureTransformerTagger(
    num_pitches=len(pitch_to_int_mapping),
    num_fingers=115,
    pitch_emb_dim=12,       # dimension for pitch embedding
    numeric_dim=5,           # e.g. onset_time, offset_time, onset_vel, offset_vel, channel
    numeric_emb_dim=32,     # project numeric feats to 128
    nhead=2,
    num_layers=2,
    dim_feedforward=128,     # example dimension in feedforward
    dropout=0.05
)

state_dict = torch.load('transformer_model_weights.pth')
transformer_model.load_state_dict(state_dict)


/tmp/ipykernel_5248/4159715045.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('transformer_model_weights.pth')


<All keys matched successfully>

In [50]:
transformer_preds, transformer_true, transformer_pred_labels, transformer_true_labels, transformer_acc = predict_piece_fingerings_multi(
    model=transformer_model,
    file_path=file_path,
    pitch_to_idx=pitch_to_int_mapping,
    finger_to_idx=finger_to_int_mapping,
    idx_to_finger=int_to_finger_mapping
)

transformer_preds

     noteID  onset_time  offset_time spelled_pitch  onset_velocity  \
0         0         0.5         1.00           C#5              80   
1         1         1.0         1.50           C#5              80   
2         2         1.5         2.00            B4              80   
3         3         2.0         4.00           C#5              80   
4         4         4.0         4.50           C#5              80   
..      ...         ...          ...           ...             ...   
390     390       284.5       286.50            A4              80   
391     391       286.5       287.00            B4              80   
392     392       287.0       287.50           F#4              80   
393     393       287.5       288.00            A4              80   
394     394       288.0       291.75            A4              80   

     offset_velocity  channel  spelled_pitch_int  
0                 80        0                113  
1                 80        0                113  
2     

array([ 0,  0,  0,  0,  0,  0,  0,  3,  0,  0,  0,  3,  0,  0,  3,  0,  0,
        3,  0,  0,  3,  3,  3,  0,  3,  3,  3,  3,  0,  0,  0,  0,  3,  0,
        0,  0,  0,  0,  3,  0,  0,  3,  3,  3,  3,  0,  0,  4,  0,  3,  3,
        3,  0,  4,  4,  3,  4,  4,  4,  4,  4,  4,  4,  4,  3,  3,  0,  1,
        0,  4,  3,  4,  3,  1,  0,  0,  0,  0,  0,  4,  4,  4,  4,  4,  4,
        4,  4,  1,  1,  0,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,
        4,  4,  4,  4,  4,  4,  4,  4,  1,  1,  0,  4,  4,  0,  4,  4,  4,
        4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  0,  4,  4,  0,  0,  1,
        0,  0,  4,  4,  0,  4,  4,  4,  0,  4,  4,  0,  4,  4,  4,  4,  4,
        0,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,
        4,  4,  1,  1,  4,  1,  1,  1,  0,  0,  1,  0,  0,  1,  0,  0,  4,
        4,  4,  4,  0,  4,  4,  4,  4,  4,  4,  0,  1,  4,  1,  1,  1,  1,
        1,  4,  1,  0,  0,  1,  1,  0,  1,  0,  0,  1,  0,  1,  1,  1,  0,
        0,  1,  0,  4,  4

# Convert any MIDI into a musicXML that can be opened in MuseScore

Replace MIDI with the OG midi you used before the text file path

Then use the list of fingerings

In [51]:
FINGERINGS_TO_USE = transformer_preds

from music21 import converter, note, chord, articulations

def add_fingerings_to_midi(midi_file, fingerings, output_xml="output.musicxml"):
    # Parse the MIDI file into a music21 stream
    score = converter.parse(midi_file)
    
    # Check if there are multiple parts
    if len(score.parts) >= 2:
        # Assume the first part is the right-hand (treble) part
        right_hand_part = score.parts[0]
    else:
        print("ERROR: Could not identify separate right-hand part in the score!")
        return
    
    # Flatten the right-hand part to iterate over its notes/chords in order
    right_hand_notes = right_hand_part.flat.getElementsByClass([note.Note, chord.Chord])
    
    # Print the counts
    print(f"Number of right-hand notes: {len(right_hand_notes)}")
    print(f"Number of fingerings: {len(fingerings)}")
    
    # Ensure the numbers match
    if len(right_hand_notes) != len(fingerings):
        print("ERROR: Number of right-hand notes and fingerings do not match!")
        return  # Stop execution if they don't match
    
    # Apply fingerings only to right-hand notes
    for i, event in enumerate(right_hand_notes):
        fingering_number = fingerings[i]
        if isinstance(event, note.Note):
            event.articulations.append(articulations.Fingering(fingering_number))
        elif isinstance(event, chord.Chord):
            # Attach the fingering to the lowest note of the chord
            event.notes[0].articulations.append(articulations.Fingering(fingering_number))
    
    # Write the score to a MusicXML file that MuseScore can open
    score.write("musicxml", fp=output_xml)
    print(f"MusicXML file saved as: {output_xml}")

# Example usage:
midi_file = "TESTY.mid"  # Replace with your MIDI file path

def reverse_dictionary(original_dict):
    reversed_dict = {}
    for key, value in original_dict.items():
        reversed_dict[value] = key
    return reversed_dict
    
reversed_fingers = reverse_dictionary(finger_to_int_mapping)
fingerings = [reversed_fingers.get(item, item) for item in FINGERINGS_TO_USE]

add_fingerings_to_midi(midi_file, fingerings, output_xml="output.musicxml")


/home/willy/Desktop/projects/smolgrad/smol/lib/python3.12/site-packages/music21/stream/base.py:3689: Music21DeprecationWarning: .flat is deprecated.  Call .flatten() instead
  return self.iter().getElementsByClass(classFilterList)


Number of right-hand notes: 395
Number of fingerings: 395
MusicXML file saved as: output.musicxml


In [52]:
!musescore output.musicxml

QSocketNotifier: Can only be used with threads started with QThread
Loading soundfont: /usr/share/mscore-2.3/sound/sf3/MuseScore_General.sf3
libpng warning: iCCP: known incorrect sRGB profile
Info at line 24 col 33: skipping 'instrument-abbreviation'
Info at line 36 col 33: skipping 'instrument-abbreviation'
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
Creating interface for ScoreView object
SoundFont(/usr/share/mscore-2.3/sound/sf3/MuseScore_General.sf3) Sample(Piano MF B1(L)) start(0) startloop(328404) endloop(478112) end(478118) smaller than SoundFont 2.04 spec chapter 7.10 recommendation
SoundFont(/usr/share/mscore-2.3/sound/sf3/MuseScore_General.sf3) Sample(Piano MF B1(R)) start(0) startloop(328404) endloop(478112) end(478118) smaller than SoundFont 2.04 spec chapter 7.10 recommendation


# how to fix messed up fingerings format-wise

in musescore go to "style", then "text", then look for "Fingering" in the left hand bar. Then set the "Vertical" field to -1.0.